# Online Retail II — Exploratory Data Analysis

This notebook provides a comprehensive EDA of the Online Retail II dataset before running the CLV pipeline.

**Sections:**
1. Data loading and overview
2. Revenue distribution
3. Customer cohort analysis
4. RFM heatmap
5. Country-level breakdown
6. Seasonality / time series


In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from pathlib import Path

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_palette('husl')

PROCESSED_DIR = Path('../data/processed')
OUTPUTS_DIR = Path('../outputs/reports')
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

print('Setup complete.')

## 1. Data Loading & Overview

In [ ]:
df = pd.read_parquet(PROCESSED_DIR / 'transactions.parquet')
print(f'Shape: {df.shape}')
print(f'Date range: {df.InvoiceDate.min().date()} → {df.InvoiceDate.max().date()}')
print(f'Unique customers: {df.CustomerID.nunique():,}')
print(f'Unique invoices:  {df.InvoiceNo.nunique():,}')
print(f'Total revenue:    £{df.Revenue.sum():,.2f}')
df.head()

In [ ]:
df.info()
df.describe()

## 2. Revenue Distribution

In [ ]:
# Per-invoice revenue
invoice_rev = df.groupby('InvoiceNo')['Revenue'].sum()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(invoice_rev.clip(upper=invoice_rev.quantile(0.99)), bins=80, color='#4f8ef7', edgecolor='white')
axes[0].set_title('Invoice Revenue Distribution (clipped at 99th pct)')
axes[0].set_xlabel('Revenue (£)')
axes[0].set_ylabel('Count')

# Log scale
axes[1].hist(np.log1p(invoice_rev), bins=80, color='#f74f4f', edgecolor='white')
axes[1].set_title('Invoice Revenue Distribution (log scale)')
axes[1].set_xlabel('log(1 + Revenue)')
plt.tight_layout()
fig.savefig(OUTPUTS_DIR / 'revenue_distribution.png', dpi=150)
plt.show()

## 3. Customer Cohort Analysis (Monthly)

In [ ]:
df['CohortMonth'] = df.groupby('CustomerID')['InvoiceDate'].transform('min').dt.to_period('M')
df['InvoiceMonth'] = df['InvoiceDate'].dt.to_period('M')
df['CohortIndex'] = (df['InvoiceMonth'] - df['CohortMonth']).apply(lambda x: x.n)

cohort_data = df.groupby(['CohortMonth', 'CohortIndex'])['CustomerID'].nunique().reset_index()
cohort_pivot = cohort_data.pivot(index='CohortMonth', columns='CohortIndex', values='CustomerID')
cohort_pct = cohort_pivot.divide(cohort_pivot.iloc[:, 0], axis=0).iloc[:, :12]

fig, ax = plt.subplots(figsize=(14, 8))
sns.heatmap(
    cohort_pct.astype(float),
    annot=True, fmt='.0%', cmap='YlOrRd_r',
    linewidths=0.5, ax=ax, vmin=0, vmax=0.5,
    cbar_kws={'label': 'Retention Rate'}
)
ax.set_title('Customer Retention Cohort Analysis (Monthly)', fontsize=13, fontweight='bold')
ax.set_xlabel('Months Since First Purchase')
ax.set_ylabel('Cohort (First Purchase Month)')
plt.tight_layout()
fig.savefig(OUTPUTS_DIR / 'cohort_analysis.png', dpi=150)
plt.show()

## 4. RFM Heatmap

In [ ]:
rfm = pd.read_parquet(PROCESSED_DIR / 'rfm_features.parquet')

# Bin into quintiles
rfm['R_score'] = pd.qcut(rfm['recency'], q=5, labels=[5,4,3,2,1])
rfm['F_score'] = pd.qcut(rfm['frequency'].rank(method='first'), q=5, labels=[1,2,3,4,5])
rfm['M_score'] = pd.qcut(rfm['monetary_value'], q=5, labels=[1,2,3,4,5])
rfm['RFM_score'] = rfm['R_score'].astype(int) + rfm['F_score'].astype(int) + rfm['M_score'].astype(int)

# R-F heatmap coloured by avg Monetary
rf_pivot = rfm.groupby(['R_score', 'F_score'])['monetary_value'].mean().unstack()

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    rf_pivot.astype(float), annot=True, fmt='.0f', cmap='Blues',
    linewidths=0.5, ax=ax,
    cbar_kws={'label': 'Avg Monetary Value (£)'}
)
ax.set_title('RFM Heatmap — Avg Monetary Value by R×F Segment', fontweight='bold')
ax.set_xlabel('Frequency Score')
ax.set_ylabel('Recency Score')
plt.tight_layout()
fig.savefig(OUTPUTS_DIR / 'rfm_heatmap.png', dpi=150)
plt.show()

## 5. Country-Level Breakdown

In [ ]:
country_rev = df.groupby('Country')['Revenue'].sum().sort_values(ascending=False).head(15)

fig, ax = plt.subplots(figsize=(12, 6))
colors = ['#2563eb' if c == 'United Kingdom' else '#93c5fd' for c in country_rev.index]
ax.barh(country_rev.index[::-1], country_rev.values[::-1], color=colors[::-1], edgecolor='white')
ax.set_title('Top 15 Countries by Revenue', fontweight='bold')
ax.set_xlabel('Total Revenue (£)')
plt.tight_layout()
fig.savefig(OUTPUTS_DIR / 'country_revenue.png', dpi=150)
plt.show()

## 6. Revenue Time Series (Monthly)

In [ ]:
monthly = df.set_index('InvoiceDate').resample('ME')['Revenue'].sum()

fig, ax = plt.subplots(figsize=(14, 5))
ax.fill_between(monthly.index, monthly.values, alpha=0.3, color='#4f8ef7')
ax.plot(monthly.index, monthly.values, color='#2563eb', linewidth=2)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.xticks(rotation=45)
ax.set_title('Monthly Revenue Trend', fontweight='bold')
ax.set_ylabel('Revenue (£)')
plt.tight_layout()
fig.savefig(OUTPUTS_DIR / 'monthly_revenue.png', dpi=150)
plt.show()

## 7. CLV Scores Analysis (post-pipeline)

In [ ]:
try:
    clv = pd.read_csv('../outputs/clv_scores.csv')
    print(clv.describe())
    print('\nTier distribution:')
    print(clv['clv_tier'].value_counts())

    tier_order = ['Bronze', 'Silver', 'Gold', 'Platinum']
    palette = {'Bronze': '#cd7f32', 'Silver': '#aaa9ad', 'Gold': '#ffd700', 'Platinum': '#e5e4e2'}

    fig, ax = plt.subplots(figsize=(10, 5))
    sns.violinplot(data=clv, x='clv_tier', y='predicted_clv_6months',
                   order=tier_order, palette=palette, ax=ax)
    ax.set_title('CLV Distribution by Tier', fontweight='bold')
    ax.set_xlabel('CLV Tier')
    ax.set_ylabel('Predicted CLV — 6 months (£)')
    plt.tight_layout()
    plt.show()
except FileNotFoundError:
    print('CLV scores not yet generated. Run the full pipeline first.')